# Item Bronze to Silver

This notebook takes item data from the Bronze layer and prepares it for the Silver layer.

At a high level, it reads raw item records, cleans and validates the data, sends rejected records to an exception table, and loads the clean item data into the Silver table.

The goal is to make item data structured, reliable, and ready for downstream use.

In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, trim, row_number, current_date, current_timestamp, sha2, concat_ws
from pyspark.sql.window import Window
import urllib

# ==========================================
# 1. PARAMETERIZATION & CONFIGURATION
# ==========================================
# Senior engineers NEVER hardcode paths or variables. We use widgets.
dbutils.widgets.text("company_key", "ABC", "Company Key")
dbutils.widgets.text("brand_key", "ABC", "Brand Key")
dbutils.widgets.text("catalog_name", "erp_lakehouse", "Catalog Name")
dbutils.widgets.text("schema_name", "silver", "Schema Name")
dbutils.widgets.text("silver_external_path", "abfss://silver@bbmanufacturingprod.dfs.core.windows.net/delta/sl_item", "Silver External Path")
dbutils.widgets.text("exception_external_path", "abfss://silver@bbmanufacturingprod.dfs.core.windows.net/delta/exception_table", "Exception Table Path")


COMPANY_KEY = dbutils.widgets.get("company_key")
BRAND_KEY = dbutils.widgets.get("brand_key")
CATALOG_NAME= dbutils.widgets.get("catalog_name")
SILVER_SCHEMA= dbutils.widgets.get("schema_name")
SILVER_PATH = dbutils.widgets.get("silver_external_path")
EXCEPTION_PATH = dbutils.widgets.get("exception_external_path")



In [0]:
# ==========================================
# 2. DATA LOADING (BRONZE LAYER)
# ==========================================
print("Reading data from Bronze Delta Table...")
bronze_df = spark.read.table(f"{CATALOG_NAME}.bronze.bz_items")

# Inject control columns early
enriched_df = bronze_df \
    .withColumn('RecordStatus', lit('0')) \
    .withColumn('CompanyKey', lit(COMPANY_KEY)) \
    .withColumn('BrandKey', lit(BRAND_KEY))


In [0]:
# ==========================================
# 3. DATA QUALITY & INTEGRITY RULES (SILVER AUDIT)
# ==========================================
# Rule 1: Identify Null or Blank IDs (Highest Priority Exception)
validated_df = enriched_df.withColumn(
    "RecordStatus",
    when((col("id").isNull()) | (trim(col("id")) == ""), lit('2')).otherwise(col("RecordStatus"))
)

# Rule 2: Identify Duplicates using Windowing (Only on valid IDs to save performance)
# Note: We order by ingestion_time or LastModifiedDateTime descending if available, to keep the latest record!
window_spec = Window.partitionBy("id", "CompanyKey", "BrandKey").orderBy(col("ingestion_time").desc())

processed_df = validated_df.withColumn("row_num", row_number().over(window_spec)) \
    .withColumn("RecordStatus", when((col("RecordStatus") == '0') & (col("row_num") > 1), lit('1')).otherwise(col("RecordStatus"))) \
    .drop("row_num")

# Cache this dataframe because we are going to split it into two independent forks (Exceptions vs Clean Data)
processed_df.cache()


In [0]:
# ==========================================
# 4. EXCEPTION HANDLING (THE DE ROUTINE)
# ==========================================
print("Processing and routing data exceptions...")

# Create the dedicated silver schema isolation layer
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SILVER_SCHEMA}")

# Ensure exception external table exists
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.{SILVER_SCHEMA}.exception_table (
    ExceptionID STRING,
    BrandKey STRING,
    RecordKey STRING,
    TableName STRING,
    ColumnName STRING,
    ExceptionDetails STRING,
    SysCreatedDate DATE,
    SysCreatedBy STRING
)
USING DELTA
LOCATION '{EXCEPTION_PATH}'
""")

# Route Duplicates (Status 1) & Nulls (Status 2)
duplicate_exceptions = processed_df.filter(col("RecordStatus") == '1') \
    .select(
        sha2(concat_ws("||", col("id"), current_timestamp()), 256).alias("ExceptionID"),
        lit(BRAND_KEY).alias("BrandKey"),
        col("id").cast("string").alias("RecordKey"),
        lit("bz_items").alias("TableName"),
        lit("").alias("ColumnName"),
        lit("Duplicate record found during deduplication").alias("ExceptionDetails"),
        current_date().alias("SysCreatedDate"),
        lit("databricks_job").alias("SysCreatedBy")
    )

null_exceptions = processed_df.filter(col("RecordStatus") == '2') \
    .select(
        sha2(concat_ws("||", col("id"), current_timestamp()), 256).alias("ExceptionID"),
        lit(BRAND_KEY).alias("BrandKey"),
        lit("UNKNOWN_KEY").alias("RecordKey"),
        lit("bz_items").alias("TableName"),
        lit("id").alias("ColumnName"),
        lit("Null or Blank primary key 'id' identified").alias("ExceptionDetails"),
        current_date().alias("SysCreatedDate"),
        lit("databricks_job").alias("SysCreatedBy")
    )

# Union and append to external exception logs
final_exceptions_df = duplicate_exceptions.unionByName(null_exceptions)
if final_exceptions_df.count() > 0:
    final_exceptions_df.write.mode("append").saveAsTable("erp_lakehouse.silver.exception_table")


In [0]:
from pyspark.sql.functions import coalesce, col, lit, when
# ==========================================
# 5. TRANSFORM CLEAN DATA FOR SILVER LAYER
# ==========================================
print("Transforming valid corporate records...")
clean_records_df = processed_df.filter(col("RecordStatus") == '0')

print("Transforming valid corporate records using optimized coalesce operations... 🚀")

final_silver_df = clean_records_df.select(
    col("CompanyKey").cast("string"),
    col("BrandKey").cast("string"),
    col("id").alias("ItemSrcID").cast("string"),
    
    # Using coalesce to handle null-fallbacks elegantly
    coalesce(col("itemCategoryId"), lit("")).alias("ProductLnSrcId").cast("string"),
    coalesce(col("baseUnitOfMeasureId"), lit("")).alias("UOMSrcId").cast("string"),
    coalesce(col("number"), lit("")).alias("ItemKey").cast("string"),
    coalesce(col("type"), lit("")).alias("ItemType").cast("string"),
    coalesce(col("displayName"), lit("")).alias("ItemDesc").cast("string"),
    coalesce(col("displayName2"), lit("")).alias("ItemAltDesc").cast("string"),
    
    # This must remain a when() block because it evaluates string pattern-matching rules, not just nulls!
    when(col("number").like("%Freight%"), lit("Freight"))
        .when(col("number").like("%ADM%"), lit("AddlChrg"))
        .when(col("type") == "Non_x002D_Inventory", lit("Non Inventory"))
        .otherwise(coalesce(col("type"), lit(""))).alias("ItemTypeDesc").cast("string"),
        
    coalesce(col("itemCategoryCode"), lit("")).alias("ProductLnKey").cast("string"),
    coalesce(col("baseUnitOfMeasureCode"), lit("")).alias("SalesUOM").cast("string"),
    
    # Numeric defaults to 0.0
    coalesce(col("unitCost"), lit(0.0)).cast("double").alias("StdUnitCost"),
    coalesce(col("unitPrice"), lit(0.0)).cast("double").alias("StdUnitPrice"),
    
    # Metadata Timestamps
    col("LastModifiedDateTime").alias("SourceUpdatedTime").cast("timestamp"),
    col("ingestion_time").alias("SysCreatedTime").cast("timestamp"),
    col("RecordStatus").cast("string")
)

display(final_silver_df)

In [0]:
# ==========================================
# 6. EXTERNAL DELTA LAKE MERGE OPERATION
# ==========================================
print("Executing Idempotent Delta Merge into Silver Layer...")

# Create External Silver Table with accurate explicit mapping schema
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.{SILVER_SCHEMA}.sl_item (
    CompanyKey STRING,
    BrandKey STRING,
    ItemSrcID STRING,
    ProductLnSrcId STRING,
    UOMSrcId STRING,
    ItemKey STRING,
    ItemType STRING,
    ItemDesc STRING,
    ItemAltDesc STRING,
    ItemTypeDesc STRING,
    ProductLnKey STRING,
    SalesUOM STRING,
    StdUnitCost DOUBLE,
    StdUnitPrice DOUBLE,
    SourceUpdatedTime TIMESTAMP,
    SysCreatedTime TIMESTAMP,
    RecordStatus STRING
)
USING DELTA
LOCATION '{SILVER_PATH}'
""")

In [0]:
# Register temporary view for the merge execution
final_silver_df.createOrReplaceTempView("final_silver_src")


In [0]:
spark.sql("""
MERGE INTO erp_lakehouse.silver.sl_item AS tgt
USING final_silver_src AS src
ON tgt.ItemSrcID = src.ItemSrcID 
   AND tgt.CompanyKey = src.CompanyKey 
   AND tgt.BrandKey = src.BrandKey
WHEN MATCHED THEN
  UPDATE SET
    tgt.ProductLnSrcId = src.ProductLnSrcId,
    tgt.UOMSrcId = src.UOMSrcId,
    tgt.ItemKey = src.ItemKey,
    tgt.ItemType = src.ItemType,
    tgt.ItemDesc = src.ItemDesc,
    tgt.ItemAltDesc = src.ItemAltDesc,
    tgt.ItemTypeDesc = src.ItemTypeDesc,
    tgt.ProductLnKey = src.ProductLnKey,
    tgt.SalesUOM = src.SalesUOM,
    tgt.StdUnitCost = src.StdUnitCost,
    tgt.StdUnitPrice = src.StdUnitPrice,
    tgt.SourceUpdatedTime = src.SourceUpdatedTime
WHEN NOT MATCHED THEN
  INSERT (
    CompanyKey, BrandKey, ItemSrcID, ProductLnSrcId, UOMSrcId, ItemKey, ItemType, 
    ItemDesc, ItemAltDesc, ItemTypeDesc, ProductLnKey, SalesUOM, StdUnitCost, StdUnitPrice, 
    SourceUpdatedTime, SysCreatedTime, RecordStatus
  )
  VALUES (
    src.CompanyKey, src.BrandKey, src.ItemSrcID, src.ProductLnSrcId, src.UOMSrcId, src.ItemKey, src.ItemType, 
    src.ItemDesc, src.ItemAltDesc, src.ItemTypeDesc, src.ProductLnKey, src.SalesUOM, src.StdUnitCost, src.StdUnitPrice, 
    src.SourceUpdatedTime, src.SysCreatedTime, src.RecordStatus
  )
""")

# Uncache data to free memory cluster resources
processed_df.unpersist()
print("Pipeline executed successfully and cleanly.")